In [1]:
import numpy as np
import pandas as pd

df=pd.read_excel("SA_Outage_till_may 26.xlsx")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
df.head()


c:\Users\AviBhandari\miniconda3\envs\dashboard\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Sr. No.,Severity,Month,Down Time,Uptime,MTTR,MTTR in \nMints,Customer Name,RFO,Fault Type,Network,Required Action / Taken,Owner,Network Incident,Transport Circuit Id,Unnamed: 15
0,1,S1,Jan-26,01-01-2026\n15:18:15\n15:20:32\n 15:23:44\n 15:25:46,01-01-2026\n15:18:41\n15:20:58\n15:24:41\n15:26:53,00:03:48,4.0,Google (5.62T)\n(GPX-1 to Kalwa),"Fiber cut in subnet C4 since 15:06 IST on 1st Jan and optical fluctuation observed in Span (Subnet C3) as per mentioned timestamps. As per the update from field support team, no activity was conducted on Subnet C3. Cause of span fluctuation is unidentified.",Fiber,MOFN,NaN,NaN,NaN,NaN,NaN
1,2,S2,Jan-26,02-01-2026\n00:44:23,02-01-2026\n04:13:23,03:29:00,209.0,Google(6*100G)\n(CHN MOFN between Netmagic to Annasalai ),"Optics Controller Coherent, Port Minor alarm was present on SFM6 Card at Chennai Netmagic location. Transponder Card was replaced and alarm got cleared.",Hardware,MOFN,NaN,NaN,NaN,"000780, 000777, 000778, 000781, 000779, 000782",NaN
2,3,S2,Jan-26,02-01-2026\n10:04:00,02-01-2026\n10:14:00,00:10:00,10.0,"MUM-DEL: Amazon(7*100G), Oracle(1*100G), Cloudflare(1*100G), Riot Games(1*10G)\nMUM-KOL: Vodafone(4*100G)\nDEL-HYD:Oracle(1*100G)",Ongoing Fiber cut in link-2 between Vermana Vadodara to Channi bridge since 09:47 hrs IST. Fiber cut at Mumbai Triangle section between IBM to GPX2 at 10:04 hrs IST. Service rerouted via alternate path.,Fiber,Smartnet,GPX-2 to IBM is shifted from UG to OPGW,NaN,NaN,"000137, 000138, 000139, 000140, 000141, 000290, 000291, 001321, 000998, 000125, 001177, 001178, 001263, 001264, 001322",NaN
3,4,S3,Jan-26,03-01-2026\n02:31:00,03-01-2026\n03:22:43,00:51:43,52.0,Riot Games (1*10G MUM-DEL Offnet Circuit),"Fiber cut in supplier TCL network. Path-1 was affected between Delhi and Vaghodia in the Mavli–Kankroli section, and Path-2 was affected between Mumbai LVSB and Delhi in the Burda–Chitara section.",Fiber,Smartnet,NaN,NaN,NaN,"003129,",NaN
4,5,S1,Jan-26,06-01-2026\n23:30:50,06-01-2026\n23:31:06,00:00:16,0.0,Microsoft MAA026\n(11.3T),Fluctuation due to JCB activity at on the route. The team was onsite and performed a chamber shifting to avoid any foreseeable damage. This is Sev-4 incident.,Fiber,MOFN,NaN,NaN,NaN,MAA026,NaN


In [7]:
from rapidfuzz import process, fuzz
import re

companies = [
    "Airtel", "Akamai", "Amazon", "Amazon LEO", "Apple",
    "Bloomberg", "Citadel", "Cloudflare", "Colt", "Edgerede", "Flipkart",
    "Google", "HGC", "IDFC", "Jump Traders", "Kuiper",
    "Meta", "Microsoft", "Netflix", "Netmagic", "NTT", "NVIDIA", "Oracle",
    "PhonePe", "Riot Games", "Sify", "TCL", "Telstra", "Verizon",
    "Vodafone", "Wells Fargo", "Zenlayer", "Zoho"
]

def is_noise_token(token):
    """Filter out numeric codes and port/speed specs."""
    digit_ratio = sum(c.isdigit() for c in token) / max(len(token), 1)
    if digit_ratio > 0.5:
        return True
    # Alphanumeric codes with at least one digit: MAA026, 100G, DEL3
    if re.fullmatch(r'[A-Z0-9]{2,}', token) and any(c.isdigit() for c in token):
        return True
    return False

def normalize(text):
    """Lowercase and remove spaces for comparison."""
    return text.lower().replace(" ", "")

def extract_companies(customer_text, company_list, threshold=85):
    if not isinstance(customer_text, str):
        return []

    # Split on delimiters including hyphen
    delimiter_tokens = re.split(r'[\n,:()\[\]*\\\-]+', customer_text)
    delimiter_tokens = [t.strip() for t in delimiter_tokens if t.strip()]

    # Further split on whitespace
    words = []
    for token in delimiter_tokens:
        words.extend(token.split())

    # Filter noise tokens
    clean_words = [w for w in words if not is_noise_token(w)]

    # Build 1, 2, 3-word candidates
    candidates = set(clean_words)
    for n in [2, 3]:
        for i in range(len(clean_words) - n + 1):
            candidates.add(" ".join(clean_words[i:i+n]))

    # Map 1: lowercase with spaces — "meta" -> "Meta"
    company_lower_map   = {c.lower(): c for c in company_list}
    company_lower_list  = list(company_lower_map.keys())

    # Map 2: no spaces — "phonePe" -> "PhonePe" (handles "Phone Pe" inputs)
    company_nospace_map  = {normalize(c): c for c in company_list}
    company_nospace_list = list(company_nospace_map.keys())

    matches = set()
    for candidate in candidates:
        # Pass 1: normal lowercase match
        result = process.extractOne(
            candidate.lower(),
            company_lower_list,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold
        )
        if result:
            matches.add(company_lower_map[result[0]])
            continue

        # Pass 2: space-stripped match — catches "Phone Pe" -> "PhonePe"
        result = process.extractOne(
            normalize(candidate),
            company_nospace_list,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold
        )
        if result:
            matches.add(company_nospace_map[result[0]])

    return sorted(matches)

In [8]:
df['Customers'] = df['Customer Name'].apply(lambda x: extract_companies(x, companies))
df[['Customer Name', 'Customers']].head(20)

,Customer Name,Customers
0,Google (5.62T)\n(GPX-1 to Kalwa),[Google]
1,Google(6*100G)\n(CHN MOFN between Netmagic to Annasalai ),"[Google, Netmagic]"
2,"MUM-DEL: Amazon(7*100G), Oracle(1*100G), Cloudflare(1*100G), Riot Games(1*10G)\nMUM-KOL: Vodafone(4*100G)\nDEL-HYD:Oracle(1*100G)","[Amazon, Cloudflare, Oracle, Riot Games, Vodafone]"
3,Riot Games (1*10G MUM-DEL Offnet Circuit),[Riot Games]
4,Microsoft MAA026\n(11.3T),[Microsoft]
5,Edgerede(4*10G MUM Metro),[Edgerede]
6,Amazon(5*100G MUM-HYD),[Amazon]
7,"MUM-CHN: Akamai(1*100G), IDFC(1*10G), Netflix(1*100G), PHONE PE(2*10G), SIFY CORP(2*10G), Telstra (3*100G), Wells Fargo(1*10G), Zenlayer(10G) CHN METRO: Flipkart(1*10G) \nBGL-CHN:Phonepe (2*10G)","[Akamai, Flipkart, IDFC, Netflix, PhonePe, Sify, Telstra, Wells Fargo, Zenlayer]"
8,"CHN Metro: Microsoft (8*100G), NTT(2*10G)\nCHN-KOL: Akamai(1*100G)","[Akamai, Microsoft, NTT]"
9,"MUM-CHN: Amazon(4*100G), Telstra(1*100G)\nCHN Metro: Microsoft(4*100G)","[Amazon, Microsoft, Telstra]"


In [9]:
df[df['Customers'].apply(lambda x: x == [])]

,Sr. No.,Severity,Month,Down Time,Uptime,MTTR,MTTR in \nMints,Customer Name,RFO,Fault Type,Network,Required Action / Taken,Owner,Network Incident,Transport Circuit Id,Unnamed: 15,Customers


In [10]:
df.drop(df[df['Customer Name'] == '4'].index, inplace=True)

In [11]:
df[df['Customers'].apply(lambda x: x == [])]

,Sr. No.,Severity,Month,Down Time,Uptime,MTTR,MTTR in \nMints,Customer Name,RFO,Fault Type,Network,Required Action / Taken,Owner,Network Incident,Transport Circuit Id,Unnamed: 15,Customers


In [12]:
df.columns


Index(['Sr. No.', 'Severity', 'Month', 'Down Time', 'Uptime', 'MTTR',
       'MTTR in \nMints', 'Customer Name', 'RFO', 'Fault Type', 'Network',
       'Required Action / Taken', 'Owner', 'Network Incident',
       'Transport Circuit Id', 'Unnamed: 15', 'Customers'],
      dtype='object')

In [20]:
print(df['Down Time'].dtype)
print(type(df['Down Time'].iloc[0]))
print(df['Down Time'].head())

datetime64[ns]
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
0   2026-01-01 15:25:46
1   2026-01-02 00:44:23
2   2026-01-02 10:04:00
3   2026-01-03 02:31:00
4   2026-01-06 23:30:50
Name: Down Time, dtype: datetime64[ns]


In [21]:
df['date'] = df['Down Time'].dt.normalize()

In [24]:
df.to_csv("parsed_data.csv", index=False)

In [25]:
df['date'].isna().sum()

np.int64(1)